In [1]:
!pip install -q -U \
    bitsandbytes\
    transformers \
    peft\
    accelerate \
    datasets \
    trl\
    sentencepiece \
    protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.2 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.2

In [19]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3

Found existing installation: protobuf 6.33.2
Uninstalling protobuf-6.33.2:
  Successfully uninstalled protobuf-6.33.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.


In [1]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


CUDA available: True
CUDA device: Tesla T4


In [2]:
# Model configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama-dolly-qlora"
FINAL_MODEL_DIR = "./tinyllama-dolly-merged"

print(f"  LOADING MODEL: {MODEL_NAME}")


  LOADING MODEL: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [3]:
# Configure 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # ✅ CHANGED: Use bfloat16 instead of float16
    bnb_4bit_use_double_quant=True,
)

print(" 4-bit Quantization Config")
print(f"   - Quantization type: NF4")
print(f"   - Compute dtype: bfloat16")  # ✅ Updated print
print(f"   - Double quantization: Enabled")


 4-bit Quantization Config
   - Quantization type: NF4
   - Compute dtype: bfloat16
   - Double quantization: Enabled


In [4]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n Tokenizer loaded: {len(tokenizer)} tokens")
print(f"   - EOS token: {tokenizer.eos_token}")
print(f"   - PAD token: {tokenizer.pad_token}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



 Tokenizer loaded: 32000 tokens
   - EOS token: </s>
   - PAD token: </s>


In [5]:
# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing for memory efficiency
model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

print(f"\n Model loaded in 4-bit (memory: {model.get_memory_footprint() / 1e9:.2f} GB)")



 Model loaded in 4-bit (memory: 0.75 GB)


In [6]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print(" Model prepared for k-bit training")


 Model prepared for k-bit training


In [7]:
# Load Dolly-15k dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(f"\n Dataset loaded: {len(dataset)} samples")
print(f"\n Dataset features: {dataset.features}")
print(f"\n Sample entry:")
print(dataset[0])



 Dataset loaded: 15011 samples

 Dataset features: {'instruction': Value('string'), 'context': Value('string'), 'response': Value('string'), 'category': Value('string')}

 Sample entry:
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [8]:
# Dataset formatting function for TinyLlama chat template
def format_dolly_dataset(sample):
    """
    Format Dolly dataset into TinyLlama chat template.

    Dolly columns:
    - instruction: The task/question
    - context: Additional context (may be empty)
    - response: The expected answer

    Target format:
    <|user|>
    {instruction}
    {context}</s>
    <|assistant|>
    {response}</s>
    """
    instruction = sample["instruction"].strip()
    context = sample["context"].strip()
    response = sample["response"].strip()

    # Combine instruction and context
    if context:
        user_message = f"{instruction}\n\nContext: {context}"
    else:
        user_message = instruction

    # Format using TinyLlama chat template
    formatted_text = f"<|user|>\n{user_message}</s>\n<|assistant|>\n{response}</s>"

    return {"text": formatted_text}

print(" Formatting function defined")


 Formatting function defined


In [9]:
# Apply formatting to dataset with truncation check
def format_and_validate(sample):
    formatted = format_dolly_dataset(sample)
    # Tokenize to check length
    tokens = tokenizer(formatted["text"], truncation=False, add_special_tokens=True)
    # If too long, truncate the response
    if len(tokens["input_ids"]) > 500:  # Leave room for prompt
        # Truncate context if present
        instruction = sample["instruction"].strip()
        context = sample["context"].strip()[:200] if sample["context"].strip() else ""
        response = sample["response"].strip()[:200]  # Truncate response

        if context:
            user_message = f"{instruction}\n\nContext: {context}"
        else:
            user_message = instruction

        formatted["text"] = f"<|user|>\n{user_message}</s>\n<|assistant|>\n{response}</s>"

    return formatted

formatted_dataset = dataset.map(
    format_and_validate,
    remove_columns=dataset.column_names,
    desc="Formatting Dolly dataset"
)

print(f"\n Dataset formatted: {len(formatted_dataset)} samples")
print(f"\n Formatted sample:")
print(formatted_dataset[0]["text"][:500] + "...")



 Dataset formatted: 15011 samples

 Formatted sample:
<|user|>
When did Virgin Australia start operating?

Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly se...


In [10]:
# # Apply formatting to dataset
# formatted_dataset = dataset.map(
#     format_dolly_dataset,
#     remove_columns=dataset.column_names,
#     desc="Formatting Dolly dataset"
# )

# print(f"\n Dataset formatted: {len(formatted_dataset)} samples")
# print(f"\n Formatted sample:")
# print(formatted_dataset[0]["text"][:500] + "...")


In [11]:
# Split dataset for training (90% train, 10% validation)
train_test_split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"\n Dataset split:")
print(f"   - Training samples: {len(train_dataset)}")
print(f"   - Validation samples: {len(eval_dataset)}")



 Dataset split:
   - Training samples: 13509
   - Validation samples: 1502


In [12]:
# LoRA configuration with rank=16
peft_config = LoraConfig(
    r=16,                              # LoRA rank (as required)
    lora_alpha=32,                     # LoRA alpha (scaling factor)
    lora_dropout=0.05,                 # Dropout probability
    bias="none",                       # Don't train bias parameters
    task_type="CAUSAL_LM",            # Task type
    target_modules=[                   # Target modules for TinyLlama
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("\n LoRA Configuration:")
print(f"   - Rank (r): {peft_config.r}")
print(f"   - Alpha: {peft_config.lora_alpha}")
print(f"   - Dropout: {peft_config.lora_dropout}")
print(f"   - Target modules: {peft_config.target_modules}")



 LoRA Configuration:
   - Rank (r): 16
   - Alpha: 32
   - Dropout: 0.05
   - Target modules: {'gate_proj', 'q_proj', 'o_proj', 'v_proj', 'down_proj', 'k_proj', 'up_proj'}


In [13]:
# Apply LoRA to model
model = get_peft_model(model, peft_config)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"\n LoRA Adapters Attached:")
    print(f"   - Trainable params: {trainable_params:,} ({100 * trainable_params / all_param:.4f}%)")
    print(f"   - Total params: {all_param:,}")

print_trainable_parameters(model)



 LoRA Adapters Attached:
   - Trainable params: 12,615,680 (2.0082%)
   - Total params: 628,221,952


In [17]:
from transformers import TrainingArguments

# Training arguments (corrected for current API)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,  # ✅ Use BF16 for consistency with quantization
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    weight_decay=0.001,
    group_by_length=True,
    report_to="none",
)

print(" Training Arguments Configured")
print(f"   - Epochs: {training_args.num_train_epochs}")
print(f"   - Batch size: {training_args.per_device_train_batch_size}")
print(f"   - Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   - Learning rate: {training_args.learning_rate}")
print(f"   - Precision: BF16")
print(f"   - Optimizer: {training_args.optim}")


 Training Arguments Configured
   - Epochs: 1
   - Batch size: 4
   - Gradient accumulation: 4
   - Effective batch size: 16
   - Learning rate: 0.0002
   - Precision: BF16
   - Optimizer: OptimizerNames.PAGED_ADAMW_8BIT


In [18]:
# Initialize SFTTrainer with latest API
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=training_args,  # All config now goes through TrainingArguments
)

print("\n SFTTrainer initialized")
print(f"   - Training samples: {len(trainer.train_dataset)}")
print(f"   - Validation samples: {len(trainer.eval_dataset)}")
print(f"   - Model type: {model.config.model_type}")


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (8674 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]


 SFTTrainer initialized
   - Training samples: 13509
   - Validation samples: 1502
   - Model type: llama


In [19]:
# Start training
print("  STARTING TRAINING")

trainer.train()
print("  TRAINING COMPLETE")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


  STARTING TRAINING


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.261800,1.487843,1.496116,2106611.000000,0.668505


  TRAINING COMPLETE


In [20]:
# Save final LoRA adapters
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n LoRA adapters saved to: {OUTPUT_DIR}")



 LoRA adapters saved to: ./tinyllama-dolly-qlora


In [21]:
# Test generation with fine-tuned model
def generate_response(instruction, context=""):
    if context:
        prompt = f"<|user|>\n{instruction}\n\nContext: {context}</s>\n<|assistant|>\n"
    else:
        prompt = f"<|user|>\n{instruction}</s>\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract only the assistant's response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[1].split("</s>")[0].strip()
    return response

print("\n Testing fine-tuned model:\n")

test_instruction = "What is machine learning?"
response = generate_response(test_instruction)
print(f"Instruction: {test_instruction}")
print(f"Response: {response}")



 Testing fine-tuned model:

Instruction: What is machine learning?
Response: Machine learning is the use of computer programs to analyze data in order to make predictions or decisions.


In [22]:
# Reload base model (without quantization for merging)
print("\n Loading base model for merging...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print("Base model loaded")
# Load LoRA adapters and merge
print("\n Merging LoRA adapters with base model...")

model_with_adapters = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = model_with_adapters.merge_and_unload()

print(" LoRA adapters merged")


`torch_dtype` is deprecated! Use `dtype` instead!



 Loading base model for merging...
Base model loaded

 Merging LoRA adapters with base model...
 LoRA adapters merged


In [23]:
# Save merged model
merged_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"\n Merged model saved to: {FINAL_MODEL_DIR}")
print(f"   - Model size: {sum(os.path.getsize(os.path.join(FINAL_MODEL_DIR, f)) for f in os.listdir(FINAL_MODEL_DIR)) / 1e9:.2f} GB")



 Merged model saved to: ./tinyllama-dolly-merged
   - Model size: 2.20 GB


# converting the model to GGUF format

In [24]:
# Clone llama.cpp repository
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt


Cloning into 'llama.cpp'...
remote: Enumerating objects: 73380, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 73380 (delta 40), reused 23 (delta 23), pack-reused 73314 (from 3)
Receiving objects: 100% (73380/73380), 251.57 MiB | 33.33 MiB/s, done.
Resolving deltas: 100% (53094/53094), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 127.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [37]:
# Build llama.cpp
# Clone llama.cpp repository
# Clean up and rebuild llama.cpp with CMake
import os
import shutil

# Remove old llama.cpp if it exists
if os.path.exists("llama.cpp"):
    shutil.rmtree("llama.cpp")

# Clone fresh copy
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt

# Build with CMake (NEW METHOD)
%cd llama.cpp

# Create build directory and compile
!cmake -B build
!cmake --build build --config Release -j$(nproc)

# Copy binary to root for easy access
!cp build/bin/llama-quantize ./llama-quantize || cp build/llama-quantize ./llama-quantize

%cd ..

# Verify the binary exists and has size
!ls -lh llama.cpp/llama-quantize

print("\n llama-quantize built successfully!")




Cloning into 'llama.cpp'...
remote: Enumerating objects: 73380, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 73380 (delta 40), reused 23 (delta 23), pack-reused 73314 (from 3)
Receiving objects: 100% (73380/73380), 251.65 MiB | 26.12 MiB/s, done.
Resolving deltas: 100% (53123/53123), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-

In [45]:
# Convert HuggingFace model to GGUF (FP16)
# Set your model directory
FINAL_MODEL_DIR = "/content/tinyllama-dolly-merged"

# Convert to FP16 GGUF
print("  CONVERTING TO FP16 GGUF")

!python3 llama.cpp/convert_hf_to_gguf.py {FINAL_MODEL_DIR}

# Check output
import os
fp16_path = f"{FINAL_MODEL_DIR}/Tinyllama-Dolly-Merged-1.1B-F16.gguf"

if os.path.exists(fp16_path):
    size = os.path.getsize(fp16_path) / 1e9
    print(f"\n FP16 GGUF created: {size:.2f} GB")
else:
    print("\n Conversion failed")



  CONVERTING TO FP16 GGUF
INFO:hf-to-gguf:Loading model: tinyllama-dolly-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-

In [46]:
# Quantize to Q4_K_M using llama-quantize

!./llama.cpp/llama-quantize \
    {FINAL_MODEL_DIR}/Tinyllama-Dolly-Merged-1.1B-F16.gguf \
    {FINAL_MODEL_DIR}/tinyllama-dolly-Q4_K_M.gguf \
    Q4_K_M

# Verify output
import os

fp16_path = "/content/tinyllama-dolly-merged/Tinyllama-Dolly-Merged-1.1B-F16.gguf"
q4km_path = "/content/tinyllama-dolly-merged/tinyllama-dolly-Q4_K_M.gguf"
# /content/tinyllama-dolly-merged/Tinyllama-Dolly-Merged-1.1B-F16.gguf


if os.path.exists(fp16_path):
    fp16_size = os.path.getsize(fp16_path) / 1e9
    print(f"\n✅ FP16 GGUF:")
    print(f"   - Path: {fp16_path}")
    print(f"   - Size: {fp16_size:.2f} GB")

if os.path.exists(q4km_path):
    q4km_size = os.path.getsize(q4km_path) / 1e9
    compression = (1 - q4km_size / fp16_size) * 100 if os.path.exists(fp16_path) else 0
    print(f"\n✅ Q4_K_M GGUF:")
    print(f"   - Path: {q4km_path}")
    print(f"   - Size: {q4km_size:.2f} GB")
    if compression > 0:
        print(f"   - Compression: {compression:.1f}% smaller")



main: build = 7548 (7ac890213)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/tinyllama-dolly-merged/Tinyllama-Dolly-Merged-1.1B-F16.gguf' to '/content/tinyllama-dolly-merged/tinyllama-dolly-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/tinyllama-dolly-merged/Tinyllama-Dolly-Merged-1.1B-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Tinyllama Dolly Merged
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32 